Import the Libraries

In [1]:
import requests
from  bs4 import BeautifulSoup
import csv
import pandas as pd
import time
from urllib.parse import urljoin
import sqlite3


Get URL and send Get request

In [2]:
url= "https://books.toscrape.com/"
response= requests.get(url)
if response.status_code == 200:
    print("Request Successful")
else:
    print("Request Failed")

Request Successful


Parse the HTML Content

In [3]:
soup= BeautifulSoup(response.text, "html.parser")
print(soup)

<!DOCTYPE html>

<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-us"> <!--<![endif]-->
<head>
<title>
    All products | Books to Scrape - Sandbox
</title>
<meta content="text/html; charset=utf-8" http-equiv="content-type"/>
<meta content="24th Jun 2016 09:29" name="created"/>
<meta content="" name="description"/>
<meta content="width=device-width" name="viewport"/>
<meta content="NOARCHIVE,NOCACHE" name="robots"/>
<!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
<!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
<link href="static/oscar/favicon.ico" rel="shortcut icon"/>
<link href="static/oscar/css/styles.css" rel="stylesheet" type="text/css"/>
<link href="s

Extract Book Details for Page 1


In [4]:
#find all 20 Books on page 1
books= soup.find_all('h3')
start_time= time.time()
books_extracted= 0
#Iterate through the books ad extract the information for each book
for book in books:
    book_url= book.find('a')['href']

    #full_book_url = url + book_url
    full_book_url = urljoin(url, book_url)

    book_response= requests.get(url + book_url)
    book_soup= BeautifulSoup(book_response.content, "html.parser")
    title= book_soup.find('h1').text
    category= book_soup.find('ul', class_='breadcrumb').find_all('a')[2].text.strip()
    rating= book_soup.find('p', class_='star-rating')['class'][1]
    price= book_soup.find('p', class_='price_color').text.strip()
    availability= book_soup.find('p', class_='availability').text.strip()
    books_extracted += 1

    end_time=time.time()
    total_time= (end_time - start_time)/60.0

    print(f"Title: {title}")
    print(f"Category: {category}")
    print(f"Rating: {rating}")
    print(f"Price: {price}")
    print(f"Availability: {availability}")
    print(f"Book URL: {full_book_url}")
    print("**************")


print(f"Number of Books Extracted {books_extracted}")
print(f"Total Time : {total_time:.4f} ")

Title: A Light in the Attic
Category: Poetry
Rating: Three
Price: £51.77
Availability: In stock (22 available)
Book URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
**************
Title: Tipping the Velvet
Category: Historical Fiction
Rating: One
Price: £53.74
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
**************
Title: Soumission
Category: Fiction
Rating: One
Price: £50.10
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/soumission_998/index.html
**************
Title: Sharp Objects
Category: Mystery
Rating: Four
Price: £47.82
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/sharp-objects_997/index.html
**************
Title: Sapiens: A Brief History of Humankind
Category: History
Rating: Five
Price: £54.23
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/sapiens-a-brief-h

Extract details for all 5 pages

In [5]:
# Create a List to hold all the book information
books_data = []

# Loop through all 5 pages
for page_num in range(1, 6):

    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")

    books = soup.find_all('h3')


    start_time = time.time()

    # Store books for the current page only
    page_books = []

    for book in books:

        book_url = book.find('a')['href']
        full_book_url = urljoin(url, book_url)


        book_response = requests.get(
            'https://books.toscrape.com/catalogue/' + book_url
        )

        book_soup = BeautifulSoup(
            book_response.content, "html.parser"
        )

        title = book_soup.find('h1').text.strip()

        category = (
            book_soup
            .find('ul', class_='breadcrumb')
            .find_all('a')[2]
            .text
            .strip()
        )

        rating = book_soup.find(
            'p', class_='star-rating'
        )['class'][1]

        price = book_soup.find(
            'p', class_='price_color'
        ).text.strip()

        availability = book_soup.find(
            'p', class_='availability'
        ).text.strip()

        book_data = [
            title,
            category,
            rating,
            price,
            availability,
            full_book_url
        ]

        # Add to the main list
        books_data.append(book_data)

        # Add to current page list
        page_books.append(book_data)

    end_time = time.time()

    total_time = (end_time - start_time) / 60.0

    # Print current page data
    print(f"\n========== PAGE {page_num} ==========")

    for i, book_data in enumerate(page_books, start=1):
        print(f"\nBook {i}")
        print(f"Title: {book_data[0]}")
        print(f"Category: {book_data[1]}")
        print(f"Rating: {book_data[2]}")
        print(f"Price: {book_data[3]}")
        print(f"Availability: {book_data[4]}")
        print(f"Book URL: {book_data[5]}")
        print("***********")

    print("\n-----------------------------")
    print(f"Page {page_num} completed.")
    print(f"{len(page_books)} books extracted from this page.")
    print(f"{len(books_data)} books extracted so far.")
    print(f"Time: {total_time:.2f} minutes")
    print("*****************************")


========== PAGE 1 ==========

Book 1
Title: A Light in the Attic
Category: Poetry
Rating: Three
Price: £51.77
Availability: In stock (22 available)
Book URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
***********

Book 2
Title: Tipping the Velvet
Category: Historical Fiction
Rating: One
Price: £53.74
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
***********

Book 3
Title: Soumission
Category: Fiction
Rating: One
Price: £50.10
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/soumission_998/index.html
***********

Book 4
Title: Sharp Objects
Category: Mystery
Rating: Four
Price: £47.82
Availability: In stock (20 available)
Book URL: https://books.toscrape.com/catalogue/sharp-objects_997/index.html
***********

Book 5
Title: Sapiens: A Brief History of Humankind
Category: History
Rating: Five
Price: £54.23
Availability: In stock (20 available)
Book U

Export the Data

In [9]:
df = pd.DataFrame(
    books_data,
    columns=[
        "Title",
        "Category",
        "Rating",
        "Price",
        "Availability",
        "Book_URL"
    ]
)
print(df.head(10))

                                               Title            Category  \
0                               A Light in the Attic              Poetry   
1                                 Tipping the Velvet  Historical Fiction   
2                                         Soumission             Fiction   
3                                      Sharp Objects             Mystery   
4              Sapiens: A Brief History of Humankind             History   
5                                    The Requiem Red         Young Adult   
6  The Dirty Little Secrets of Getting Your Dream...            Business   
7  The Coming Woman: A Novel Based on the Life of...             Default   
8  The Boys in the Boat: Nine Americans and Their...             Default   
9                                    The Black Maria              Poetry   

  Rating   Price             Availability  \
0  Three  £51.77  In stock (22 available)   
1    One  £53.74  In stock (20 available)   
2    One  £50.10  In stock (

In [10]:
print(df["Book_URL"].head())


0    https://books.toscrape.com/catalogue/a-light-i...
1    https://books.toscrape.com/catalogue/tipping-t...
2    https://books.toscrape.com/catalogue/soumissio...
3    https://books.toscrape.com/catalogue/sharp-obj...
4    https://books.toscrape.com/catalogue/sapiens-a...
Name: Book_URL, dtype: object


Verify Book URLs After DataFrame Conversion

In [11]:
#print(df["Book_URL"].iloc[0])

In [12]:
for url in df["Book_URL"]:
    print(url)

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
https://books.toscrape.com/catalogue/soumission_998/index.html
https://books.toscrape.com/catalogue/sharp-objects_997/index.html
https://books.toscrape.com/catalogue/sapiens-a-brief-history-of-humankind_996/index.html
https://books.toscrape.com/catalogue/the-requiem-red_995/index.html
https://books.toscrape.com/catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html
https://books.toscrape.com/catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html
https://books.toscrape.com/catalogue/the-boys-in-the-boat-nine-americans-and-their-epic-quest-for-gold-at-the-1936-berlin-olympics_992/index.html
https://books.toscrape.com/catalogue/the-black-maria_991/index.html
https://books.toscrape.com/catalogue/starving-hearts-triangular-trade-trilogy-1_990/index.html
https://books.toscr

save to csv

In [13]:
df.to_csv("books_scraped.csv",index=False)
print("Data Saved")

Data Saved


Cleaning and Preprocessing Data





In [14]:
df = pd.read_csv("/content/books_scraped.csv")


In [15]:
df.head()

,Title,Category,Rating,Price,Availability,Book_URL
0,A Light in the Attic,Poetry,Three,£51.77,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Historical Fiction,One,£53.74,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Fiction,One,£50.10,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Mystery,Four,£47.82,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,History,Five,£54.23,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...


Check Data Types and Remove Unnecessary Column


In [16]:
# Check data types of all columns
print(df.dtypes)

Title           object
Category        object
Rating          object
Price           object
Availability    object
Book_URL        object
dtype: object


In [17]:
df = df.drop("Category", axis=1)

In [18]:
df.head()

,Title,Rating,Price,Availability,Book_URL
0,A Light in the Attic,Three,£51.77,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,One,£53.74,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,One,£50.10,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Four,£47.82,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Five,£54.23,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...


Convert Book Ratings from Text to Numeric Values

In [20]:
df["Rating"]

,Rating
0,Three
1,One
2,One
3,Four
4,Five
...,...
95,Two
96,One
97,Two
98,Five


In [21]:
df["Rating"].unique()

array(['Three', 'One', 'Four', 'Five', 'Two'], dtype=object)

In [22]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["Rating"] = df["Rating"].map(rating_map)

In [23]:
print(df["Rating"])
print(df["Rating"].dtype)

0     3
1     1
2     1
3     4
4     5
     ..
95    2
96    1
97    2
98    5
99    4
Name: Rating, Length: 100, dtype: int64
int64


In [24]:
df.head()

,Title,Rating,Price,Availability,Book_URL
0,A Light in the Attic,3,£51.77,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,1,£53.74,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,1,£50.10,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,4,£47.82,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,5,£54.23,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...


Convert Price from Text to Numeric Values

In [25]:
df["Price"]

,Price
0,£51.77
1,£53.74
2,£50.10
3,£47.82
4,£54.23
...,...
95,£19.92
96,£40.11
97,£53.90
98,£35.67


In [26]:
df["Price"] = df["Price"].str.replace("£", "", regex=False).astype(float)

In [27]:
df["Price"].head()

,Price
0,51.77
1,53.74
2,50.10
3,47.82
4,54.23


In [28]:
df.head()

,Title,Rating,Price,Availability,Book_URL
0,A Light in the Attic,3,51.77,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,1,53.74,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,1,50.10,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,4,47.82,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,5,54.23,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...


Extract Stock Status from Availability

In [29]:
df["Availability"]

,Availability
0,In stock (22 available)
1,In stock (20 available)
2,In stock (20 available)
3,In stock (20 available)
4,In stock (20 available)
...,...
95,In stock (16 available)
96,In stock (16 available)
97,In stock (16 available)
98,In stock (16 available)


In [30]:
df['Availability'].value_counts()

,count
Availability,
In stock (16 available),56
In stock (19 available),23
In stock (18 available),11
In stock (17 available),5
In stock (20 available),4
In stock (22 available),1


In [31]:
df['Availability'].unique()

array(['In stock (22 available)', 'In stock (20 available)',
       'In stock (19 available)', 'In stock (18 available)',
       'In stock (17 available)', 'In stock (16 available)'], dtype=object)

In [32]:
df["in_stock"] = df["Availability"].str.contains("In stock")

In [33]:
df['in_stock'].value_counts()

,count
in_stock,
True,100


In [34]:
df['in_stock'].unique()

array([ True])

In [35]:
df.head()

,Title,Rating,Price,Availability,Book_URL,in_stock
0,A Light in the Attic,3,51.77,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...,True
1,Tipping the Velvet,1,53.74,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...,True
2,Soumission,1,50.10,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...,True
3,Sharp Objects,4,47.82,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...,True
4,Sapiens: A Brief History of Humankind,5,54.23,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...,True


Remove the Original Availability Column

In [36]:
df = df.drop(columns=["Availability"])

In [37]:
df.head()

,Title,Rating,Price,Book_URL,in_stock
0,A Light in the Attic,3,51.77,https://books.toscrape.com/catalogue/a-light-i...,True
1,Tipping the Velvet,1,53.74,https://books.toscrape.com/catalogue/tipping-t...,True
2,Soumission,1,50.10,https://books.toscrape.com/catalogue/soumissio...,True
3,Sharp Objects,4,47.82,https://books.toscrape.com/catalogue/sharp-obj...,True
4,Sapiens: A Brief History of Humankind,5,54.23,https://books.toscrape.com/catalogue/sapiens-a...,True


Check Final Data Types and Save Cleaned Data

In [38]:
print(df.dtypes)

Title        object
Rating        int64
Price       float64
Book_URL     object
in_stock       bool
dtype: object


In [39]:
df.to_csv("books_scraped_cleaned.csv", index=False)

print("Cleaned data saved successfully!")

Cleaned data saved successfully!


Load the CSV into SQLite

In [40]:
# 1. Connect to SQLite database
conn = sqlite3.connect("books.db")

In [41]:
# 2. Load CSV
df = pd.read_csv("books_scraped_cleaned.csv")

In [42]:
# 3. Save CSV data into SQLite table
df.to_sql("books", conn, if_exists="replace", index=False)

print("CSV loaded into SQLite successfully!")

CSV loaded into SQLite successfully!


In [43]:
print(pd.read_sql_query("PRAGMA table_info(books);", conn))

   cid      name     type  notnull dflt_value  pk
0    0     Title     TEXT        0       None   0
1    1    Rating  INTEGER        0       None   0
2    2     Price     REAL        0       None   0
3    3  Book_URL     TEXT        0       None   0
4    4  in_stock  INTEGER        0       None   0


1. Average price for each rating


In [44]:
query = """
    SELECT Rating, AVG(Price) AS Avg_Price
    FROM books
    GROUP BY Rating
    ORDER BY Rating;
"""

result = pd.read_sql_query(query, conn)
print(result)

   Rating  Avg_Price
0       1  35.519545
1       2  35.909474
2       3  36.839091
3       4  33.981667
4       5  30.012105


2. The 5 most expensive books rated 4 or 5


In [45]:
query = """
    SELECT Title, Rating, Price
    FROM books
    WHERE Rating IN (4, 5)
    ORDER BY Price DESC
    LIMIT 5;
"""

result = pd.read_sql_query(query, conn)
print(result)

                                               Title  Rating  Price
0       The Death of Humanity: and the Case for Life       4  58.11
1                                The Past Never Ends       4  56.50
2              Sapiens: A Brief History of Humankind       5  54.23
3  Scott Pilgrim's Precious Little Life (Scott Pi...       5  52.29
4                                Behind Closed Doors       4  52.22


3. How many books are out of stock, per rating


In [56]:
##This two cells only to check that the third quiry work without any Error
# test_query = """
# INSERT INTO books (Title, Rating, Price, Book_URL, in_stock)
# VALUES ('Test Out of Stock Book', 4, 100.00, 'test_url', 0);
# """

# conn.execute(test_query)
# conn.commit()

In [51]:
# delete_query = """
# DELETE FROM books
# WHERE Title IN ('Test Out of Stock Book', 'Test Out of Stock Book 2');
# """

# conn.execute(delete_query)
# conn.commit()

# print("Test rows deleted successfully!")

Test rows deleted successfully!


In [52]:
query = """
    SELECT Rating, COUNT(*) AS Out_Of_Stock_Count
    FROM books
    WHERE in_stock = 0
    GROUP BY Rating
    ORDER BY Rating;
"""

result = pd.read_sql_query(query, conn)
print(result)

Empty DataFrame
Columns: [Rating, Out_Of_Stock_Count]
Index: []


Check if There Are No Out-of-Stock Books

In [47]:
print(df["in_stock"].value_counts())

in_stock
True    100
Name: count, dtype: int64


In [48]:
query = """
SELECT in_stock, COUNT(*) AS Count
FROM books
GROUP BY in_stock;
"""

result = pd.read_sql_query(query, conn)
print(result)

   in_stock  Count
0         1    100


In [53]:
query_check = "SELECT DISTINCT in_stock FROM books;"
print(pd.read_sql_query(query_check, conn))

   in_stock
0         1


In [54]:
query_in_stock = "SELECT COUNT(*) AS Total_In_Stock FROM books WHERE in_stock = 1;"
print(pd.read_sql_query(query_in_stock, conn))

   Total_In_Stock
0             100


Create and Save SQL Queries to a File

In [55]:
with open("queries.sql", "w", encoding="utf-8") as file:
    file.write("")

In [58]:
queries = """
-- 1. Average price for each rating
SELECT Rating, AVG(Price) AS Avg_Price
FROM books
GROUP BY Rating
ORDER BY Rating;

-- 2. The 5 most expensive books rated 4 or 5
SELECT Title, Rating, Price
FROM books
WHERE Rating IN (4, 5)
ORDER BY Price DESC
LIMIT 5;

-- 3. How many books are out of stock, per rating
SELECT Rating, COUNT(*) AS Out_Of_Stock_Count
FROM books
WHERE in_stock = 0
GROUP BY Rating
ORDER BY Rating;
"""

In [59]:
with open("queries.sql", "w", encoding="utf-8") as file:
    file.write(queries)

Read and Verify SQL Queries File

In [60]:
with open("queries.sql", "r", encoding="utf-8") as file:
    queries = file.read()

print(queries)


-- 1. Average price for each rating
SELECT Rating, AVG(Price) AS Avg_Price
FROM books
GROUP BY Rating
ORDER BY Rating;

-- 2. The 5 most expensive books rated 4 or 5
SELECT Title, Rating, Price
FROM books
WHERE Rating IN (4, 5)
ORDER BY Price DESC
LIMIT 5;

-- 3. How many books are out of stock, per rating
SELECT Rating, COUNT(*) AS Out_Of_Stock_Count
FROM books
WHERE in_stock = 0
GROUP BY Rating
ORDER BY Rating;



In [61]:
conn.close()